### Duel output UNet on pennate diatoms - cell infection classification
### Rayna Hamilton
### May 7, 2025

We will have trained a UNet neural network to detect the locations of pennate diatom cells in strained microscopy images in 1_train_unet.ipynb and separated out likely unique cells in 3_split_blobs.ipynb.  Let's create separate images for each unique cell found and calculate the infection ratio for each cell based on the proportion of red and blue pixels.  Given the high amount of background in many images, I am also subtracting the mean background red and blue pixel value from the cell bodies before this calculation.  This isn't a perfect method, but further accuracy will require annotating the infection status of the individual cells and training a convolutional neural network.

In [1]:
import pandas as pd
import numpy as np
import os

import matplotlib.pyplot as plt
# %matplotlib inline

from skimage.io import imread, imshow
import os
import math
from math import sqrt
import cv2
from sklearn.metrics import r2_score
import seaborn as sns

In [2]:
#Read in data
train_images,test_images=[val for val in os.listdir("../../data/train_images/") if val.endswith('jpg')],[val for val in os.listdir("../../data/test_images/") if val.endswith('jpg')]
train_masks,test_masks=[],[]
train_unique_masks,test_unique_masks=[],[]
for file in os.listdir("../../predictions/unique_cell_masks/"):
    if file in train_images:
        train_unique_masks.append(file)
    elif file in test_images:
        test_unique_masks.append(file)

In [3]:
len(train_unique_masks),len(test_unique_masks)

(26, 41)

In [4]:
cell_counts=pd.read_csv("../../cell_counts.csv")
cell_counts.head()
cell_counts[["full_filename"]]=cell_counts[["filename"]]
for row in cell_counts.index:
    for train_image in train_unique_masks:
        if train_image.startswith(cell_counts.at[row,"filename"].replace(".jpg","")):
            cell_counts.at[row,"full_filename"]=train_image
    for test_image in test_unique_masks:
        if test_image.startswith(cell_counts.at[row,"filename"].replace(".jpg","")):
            cell_counts.at[row,"full_filename"]=test_image
    if not cell_counts.at[row,"full_filename"].endswith(".jpg"):
            cell_counts.at[row,"full_filename"]=cell_counts.at[row,"full_filename"]+".jpg"

In [5]:
images_to_add=pd.DataFrame({'filename':[val for val in test_images if val not in [val+'-C.jpg' for val in cell_counts.filename]],'dataset':['none']*(len(test_images)+len(train_images)-len(cell_counts.index)),'cell_count':[-1]*(len(test_images)+len(train_images)-len(cell_counts.index)),'full_filename':[val for val in test_images if val not in [val+'-C.jpg' for val in cell_counts.filename]]})

In [6]:
cell_counts=pd.concat([cell_counts,images_to_add])

In [7]:
cell_counts.loc[cell_counts.dataset=="none","dataset"]="test"

In [8]:
cell_counts.index=[val for val in range(0,len(cell_counts.index))]

In [9]:
#Read in coordinates of predicted cells as determined in 3_split_blobs.ipynb
images,coordinates=dict(),dict()
for row in cell_counts.index:
    image=cell_counts.at[row,'full_filename']
    images[image]=imread(f"../../data/{cell_counts.at[row,'dataset']}_images/{cell_counts.at[row,'full_filename']}")
    coordinates[image]=dict()
    for line in open("../../predictions/tsvs/"+cell_counts.at[row,'full_filename'].replace('.jpg','.tsv')):
        if not line.startswith('cell_number'):
            cell_number,cell_coordinates=line.strip().split("\t")
            coordinates[image][cell_number]=dict()
            for coordinate_pair in cell_coordinates.split(';'):
                if "," in coordinate_pair:
                    coordinates[image][cell_number][coordinate_pair]=""

First, let's output images with each unique cell outlined and numbered.  This will be helpful for visualization and subsequent annotation.

In [10]:
min_x_range,min_y_range=50,50 #minimum length and width of a single-cell image 
outlined_images=dict()
for image in images:
    outlined_images[image]=np.copy(images[image])
    all_cell_coordinates=dict()
    for cell_number, cell_coordinates in coordinates[image].items():
        for coordinate_pair in cell_coordinates.keys():
                all_cell_coordinates[coordinate_pair]=""

    #get the x and y axis range of each individual cell
    if not os.path.exists(f"../../predictions/outlined_images/{image}"):
        for cell_number, cell_coordinates in coordinates[image].items():
            min_x,min_y,max_x,max_y=10000,10000,-1,-1 #values which will be replaced by the x and y axis range of a single-cell image
            for coordinate_pair in cell_coordinates.keys():
                x,y=int(coordinate_pair.split(",")[0]),int(coordinate_pair.split(",")[1])
                if x<min_x:
                    min_x=x
                if x>max_x:
                    max_x=x
                if y<min_y:
                    min_y=y
                if y>max_y:
                    max_y=y

            #outline cells and label their cell numbers
            labelled=False #whether the cell number has already been placed next to the cell in the annotated image
            for x in range(min_x,max_x):
                for y in range(min_y,max_y):
                    #find each coordinate that is not part of a cell but is directly next to a cell - these are the points that we will turn white to create an outline
                    if f"{x},{y}" not in all_cell_coordinates.keys():
                        if f"{x-1},{y}" in cell_coordinates.keys() or f"{x+1},{y}" in cell_coordinates.keys() or f"{x},{y-1}" in cell_coordinates.keys()  or f"{x},{y+1}" in cell_coordinates.keys():
                            outlined_images[image][x,y]=255
                    if not labelled and f"{x},{y}" in cell_coordinates.keys():
                        labelled=True
                        cv2.putText(outlined_images[image],str(cell_number),(min(max(y,10),1340),min(max(x,20),1050)),cv2.FONT_HERSHEY_SIMPLEX,1,(255,255,255),3)
        cv2.imwrite(f"../../predictions/outlined_images/{image}",cv2.cvtColor(outlined_images[image], cv2.COLOR_RGB2BGR))    
    print(f"Done with {image}.")

Done with THN_A_I_15_1-20240717-C18.jpg.
Done with THN_A_I_15_1-20240724-C15.jpg.
Done with THN_A_I_22_1-20240717-C45.jpg.
Done with THN_A_I_22_3-20240724-C24.jpg.
Done with THN_B_I_15_5-20240726-C26.jpg.
Done with THN_C_I_15_3-20240724-C34.jpg.
Done with THN_C_I_15_4-20240720-C45.jpg.
Done with THN_C_I_22_2-20240720-C12.jpg.
Done with THN_STARTER_1-20240716-C27.jpg.
Done with THN_Starter_2-20240711-C26.jpg.
Done with THN_STARTER_4-20240716-C30.jpg.
Done with THN_Starter_5-20240711-C24.jpg.
Done with AS_A_I_15_4-20240720-C28.jpg.
Done with AS_A_I_22_4-20240718-C15.jpg.
Done with AS_A_I_22_5-20240718-C13.jpg.
Done with AS_B_I_15_4-20240724-C29.jpg.
Done with AS_B_I_22_5-20240720-C36.jpg.
Done with AS_B_I_22_6-20240717-C10.jpg.
Done with AS_B_U_15_1-20240726-C30.jpg.
Done with AS_C_I_15_2-20240726-C21.jpg.
Done with AS_C_I_15_3-20240724-C32.jpg.
Done with AS_C_U_15_3-20240717-C21.jpg.
Done with AS_Starter_1-20240711-C44.jpg.
Done with AS_STARTER_2-20240716-C33.jpg.
Done with AS_Starter_3

Let's also output images of each unique cell, either with the cell of interest outlined or all other cells in the image slice blacked out.  The latter format is less visually intuitive, but is better for subsequent infection ratio calculation as it prevents other cells from influencing the background pixel value calculation.

In [11]:
min_x_range,min_y_range=300,300
outlined_images=dict()
for image in images:
    outlined_images[image]=np.copy(images[image])
    all_cell_coordinates=dict()
    for cell_number, cell_coordinates in coordinates[image].items():
        for coordinate_pair in cell_coordinates.keys():
                all_cell_coordinates[coordinate_pair]=""
    for cell_number, cell_coordinates in coordinates[image].items():

        #get the x and y axis range of each individual cell
        min_x,min_y,max_x,max_y=10000,10000,-1,-1
        for coordinate_pair in cell_coordinates.keys():
            x,y=int(coordinate_pair.split(",")[0]),int(coordinate_pair.split(",")[1])
            if x<min_x:
                min_x=x
            if x>max_x:
                max_x=x
            if y<min_y:
                min_y=y
            if y>max_y:
                max_y=y

        # we also extend the single-cell image a bit if the image is very short or skinny because the cell is not diagonal - this is important as it ensures that we still have some surrounding background in the sliced image
        if (max_x-min_x)<min_x_range:
            shift=int((min_x_range-max_x+min_x)/2)
            min_x=max(min_x-shift,0)
            max_x=min(max_x+shift,1040)
        if (max_y-min_y)<min_y_range:
            shift=int((min_y_range-max_y+min_y)/2)
            min_y=max(0,min_y-shift)
            max_y=min(max_y+shift,1388)

        #produce single-cell images for each cell range
        if not os.path.exists(f"../../predictions/outlined_cells/{image.replace('.jpg','_cell_'+str(cell_number)+'_'+str(min_x)+','+str(min_y)+'.jpg')}"):
            subsetted_image=np.copy(images[image][min_x:max_x,min_y:max_y])
            outlined_subsetted_image=np.copy(images[image][min_x:max_x,min_y:max_y])
            for x in range(min_x,max_x):
                for y in range(min_y,max_y):
                    if f"{x},{y}" not in all_cell_coordinates.keys():
                        if f"{x-1},{y}" in cell_coordinates.keys() or f"{x+1},{y}" in cell_coordinates.keys() or f"{x},{y-1}" in cell_coordinates.keys()  or f"{x},{y+1}" in cell_coordinates.keys():
                            outlined_subsetted_image[x-min_x,y-min_y]=255
                    else:
                        if f"{x},{y}" not in cell_coordinates.keys(): # if a coordinate is in all_cell_coordinates but is not in cell_coordinates, it is part of another cell.  We will set these pixels to black as we do not want other cells to be part of the background calculation
                            subsetted_image[x-min_x,y-min_y]=0
            cv2.imwrite(f"../../predictions/outlined_cells/{image.replace('.jpg','_cell_'+str(cell_number)+'_'+str(min_x)+','+str(min_y)+'.jpg')}",cv2.cvtColor(outlined_subsetted_image, cv2.COLOR_RGB2BGR))
            cv2.imwrite(f"../../predictions/single_cells/{image.replace('.jpg','_cell_'+str(cell_number)+'_'+str(min_x)+','+str(min_y)+'.jpg')}",cv2.cvtColor(subsetted_image, cv2.COLOR_RGB2BGR))
    print(f"Done with {image}.")

Done with THN_A_I_15_1-20240717-C18.jpg.
Done with THN_A_I_15_1-20240724-C15.jpg.
Done with THN_A_I_22_1-20240717-C45.jpg.
Done with THN_A_I_22_3-20240724-C24.jpg.
Done with THN_B_I_15_5-20240726-C26.jpg.
Done with THN_C_I_15_3-20240724-C34.jpg.
Done with THN_C_I_15_4-20240720-C45.jpg.
Done with THN_C_I_22_2-20240720-C12.jpg.
Done with THN_STARTER_1-20240716-C27.jpg.
Done with THN_Starter_2-20240711-C26.jpg.
Done with THN_STARTER_4-20240716-C30.jpg.
Done with THN_Starter_5-20240711-C24.jpg.
Done with AS_A_I_15_4-20240720-C28.jpg.
Done with AS_A_I_22_4-20240718-C15.jpg.
Done with AS_A_I_22_5-20240718-C13.jpg.
Done with AS_B_I_15_4-20240724-C29.jpg.
Done with AS_B_I_22_5-20240720-C36.jpg.
Done with AS_B_I_22_6-20240717-C10.jpg.
Done with AS_B_U_15_1-20240726-C30.jpg.
Done with AS_C_I_15_2-20240726-C21.jpg.
Done with AS_C_I_15_3-20240724-C32.jpg.
Done with AS_C_U_15_3-20240717-C21.jpg.
Done with AS_Starter_1-20240711-C44.jpg.
Done with AS_STARTER_2-20240716-C33.jpg.
Done with AS_Starter_3

Now we can calculate infection ratios for each unique cell.  Here I am defining infection ratio by first subtracting the mean red and blue pixel values in the background from the cell body, then calculating sum(blue_pixels)/(sum(red_pixels)+sum(blue_pixels)).  We will probably make a more sophisticated calculation approach later once I have access to some training data - cells with their infection status labelled.

In [12]:
infection_ratios=dict()
for file in os.listdir("../../predictions/single_cells/"):
    key="_".join(file.split("_")[:-3])+".jpg"
    cell_number=file.split("_")[-2]
    x_shift,y_shift=int(file.split("_")[-1].split(",")[0]),int(file.split("_")[-1].split(",")[1].replace(".jpg","")) #get coordinates of top-left corner of the single-cell image within the full image - this is needed to compare to our dictionary of cell coordinates
    image=imread(f"../../predictions/single_cells/{file}")
    cell_coordinates=coordinates[key][cell_number]
    red_pixels,blue_pixels,background_red_pixels,background_blue_pixels=[],[],[],[] #sum of red and blue pixel values in the cell and surrounding background
    for x in range(len(image)):
        for y in range(len(image[0])):
            red,blue=image[x,y,0],image[x,y,2]
            if f"{x+x_shift},{y+y_shift}" in cell_coordinates.keys():
                blue_pixels.append(blue)
                red_pixels.append(red)
            else:
                background_blue_pixels.append(blue)
                background_red_pixels.append(red)
    mean_background_blue,mean_background_red=np.mean(background_blue_pixels),np.mean(background_red_pixels)
    blue_pixels=np.float64(blue_pixels)
    red_pixels=np.float64(red_pixels)
    
    #subtract mean background blue and red pixel value from the cell body, set negative values to 0, and calculate infection ratio
    blue_pixels=blue_pixels-mean_background_blue
    red_pixels=red_pixels-mean_background_red
    blue_pixels[blue_pixels<0]=0
    red_pixels[red_pixels<0]=0
    infection_ratios[file]=sum(blue_pixels)/(sum(blue_pixels)+sum(red_pixels))
            

For now, I will define an infection ratio greater than 0.3 as an infected cell, and an infection ratio greater than 0.7 as a late infected/definitely dead cell.

In [13]:
partially_infected_threshold,fully_infected_threshold=0.3,0.7
infection_ratios_by_image=dict()
output=open("../../predictions/infection_ratios_by_cell.csv","w")
output.write("image,cell_number,infection_ratio\n")
for cell,ratio in infection_ratios.items():
    image_key=("-".join(cell.split("-")[:2]))
    if image_key not in infection_ratios_by_image:
        infection_ratios_by_image[image_key]=[]
    infection_ratios_by_image[image_key].append(ratio)
    output.write(f"{image_key},{cell.split('_')[-2]},{ratio}\n")
output.close()

In [14]:
cell_counts.index=["-".join(val.split("-")[:2]) for val in list(cell_counts.filename)]

Now that we have the infection ratios for each cells, we can calculate the counts of each cell type in each image.

In [15]:
cell_counts["mean_infection_ratio"]=0.0
cell_counts["uninfected_cells"]=0
cell_counts["partially_infected_cells"]=0
cell_counts["late_infected_cells"]=0

In [16]:
for key,ratios in infection_ratios_by_image.items():
    cell_counts.at[key,"mean_infection_ratio"]=np.mean(ratios)
    for ratio in ratios:
        if ratio>=partially_infected_threshold:
            if ratio>=fully_infected_threshold:
                cell_counts.at[key,"late_infected_cells"]+=1
            else:
                cell_counts.at[key,"partially_infected_cells"]+=1
        else:
            cell_counts.at[key,"uninfected_cells"]+=1

In [17]:
cell_counts.head()

,filename,dataset,cell_count,full_filename,mean_infection_ratio,uninfected_cells,partially_infected_cells,late_infected_cells
THN_A_I_15_1-20240717,THN_A_I_15_1-20240717,train,18,THN_A_I_15_1-20240717-C18.jpg,0.071051,18,0,0
THN_A_I_15_1-20240724,THN_A_I_15_1-20240724,train,15,THN_A_I_15_1-20240724-C15.jpg,0.639299,3,4,7
THN_A_I_22_1-20240717,THN_A_I_22_1-20240717,train,45,THN_A_I_22_1-20240717-C45.jpg,0.082938,34,1,0
THN_A_I_22_3-20240724,THN_A_I_22_3-20240724,train,24,THN_A_I_22_3-20240724-C24.jpg,0.326930,14,3,6
THN_B_I_15_5-20240726,THN_B_I_15_5-20240726,train,26,THN_B_I_15_5-20240726-C26.jpg,0.599922,5,11,10


In [18]:
cell_counts[["mean_infection_ratio","uninfected_cells","partially_infected_cells","late_infected_cells"]].to_csv("../../predictions/infection_ratios_by_image.csv")